# Allocation bands across synthetic market scenarios

This notebook compares constrained maximum-Sharpe portfolios across two generated covariance scenarios and three growth-asset allocation bands. The labels describe constraints, not investor recommendations.

In [ ]:
import numpy as np
import pandas as pd

from portfolio_optimizer import (
    StaticOptimizer,
    create_style_constraints,
    exponential_utility,
    portfolio_performance,
)

In [ ]:
rng = np.random.default_rng(42)
asset_names = [f"Asset {index + 1}" for index in range(10)]
growth_assets = asset_names[:6]
expected_returns_stable = rng.uniform(0.05, 0.12, len(asset_names))
expected_returns_volatile = expected_returns_stable + rng.uniform(-0.02, 0.04, len(asset_names))

stable_factor = rng.normal(size=(len(asset_names), len(asset_names)))
volatile_factor = rng.normal(size=(len(asset_names), len(asset_names)))
covariance_stable = stable_factor @ stable_factor.T * 0.01
covariance_volatile = volatile_factor @ volatile_factor.T * 0.025
risk_free_rate = 0.02

In [ ]:
allocation_bands = [
    (0.25, 0.35, "Growth band 25–35%"),
    (0.65, 0.75, "Growth band 65–75%"),
    (0.85, 0.95, "Growth band 85–95%"),
]
scenarios = {
    "Lower covariance scale": (expected_returns_stable, covariance_stable),
    "Higher covariance scale": (expected_returns_volatile, covariance_volatile),
}
bounds = tuple((0.0, 0.40) for _ in asset_names)
rows = []

for scenario_name, (expected_returns, covariance) in scenarios.items():
    optimizer = StaticOptimizer(expected_returns, covariance)
    for lower, upper, band_name in allocation_bands:
        constraints = create_style_constraints(
            asset_names,
            growth_assets,
            lower=lower,
            upper=upper,
        )
        result = optimizer.maximize_sharpe(
            risk_free_rate=risk_free_rate,
            bounds=bounds,
            constraints=constraints,
        )
        expected_return, volatility = portfolio_performance(
            result.x,
            optimizer.expected_returns,
            optimizer.covariance,
        )
        rows.append({
            "Scenario": scenario_name,
            "Allocation band": band_name,
            "Return": expected_return,
            "Volatility": volatility,
            "Sharpe": (expected_return - risk_free_rate) / volatility,
            "CARA utility": exponential_utility(expected_return, volatility),
            "Max violation": result.max_constraint_violation,
        })

In [ ]:
results = pd.DataFrame(rows).set_index(["Scenario", "Allocation band"])
formatted = results.copy()
formatted["Return"] = formatted["Return"].map(lambda value: f"{value:.2%}")
formatted["Volatility"] = formatted["Volatility"].map(lambda value: f"{value:.2%}")
formatted["Sharpe"] = formatted["Sharpe"].map(lambda value: f"{value:.4f}")
formatted["CARA utility"] = formatted["CARA utility"].map(lambda value: f"{value:.4f}")
formatted["Max violation"] = formatted["Max violation"].map(lambda value: f"{value:.2e}")
display(formatted)